# Multi-Instance Gallery Retrieval Demo (Clean)

Evaluation-only notebook for the scenario:

- Registration images: single-object only
- Gallery images: can have multiple objects
- Image score: max over instance similarities
- Optional synthetic multi-instance gallery for controlled testing
- Leakage control: registration samples excluded from search pool


In [1]:
import os
import sys
import time
import json
import random
import pickle
import hashlib
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import ipywidgets as widgets
from IPython.display import display

ROOT = Path('/workspace/PoC/dogface_fastapi_poc_qdrant')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.core.config import Settings
from app.ml.embedder import Embedder
from app.ml.cropper import NormalizedBBox, pad_bbox, crop_from_bbox

from ultralytics import YOLO

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [2]:
# ------------------------
# Configuration
# ------------------------
DATASET_ROOT = Path('/data/dogfacenet/dataset/DogFaceNet_large/images_2')

YOLO_WEIGHTS_DIR = Path('/workspace/PoC/dogface_fastapi_poc_qdrant/weights/yolo')
EMBED_HF_ROOT = Path('/workspace/PoC/dogface_fastapi_poc_qdrant/weights')
EMBED_LOCAL_MIEWID = Path('/workspace/PoC/dogface_fastapi_poc_qdrant/weights/models--conservationxlabs--miewid-msv3')

BODY_DET_WEIGHTS = YOLO_WEIGHTS_DIR / 'yolo26x.pt'

PRETRAINED_ONLY = True
MODEL_NAME = 'miewid'
DEVICE = 'cuda:0'  # or 'cpu'

YOLO_IMG_SIZE = 640
YOLO_CONF = 0.25
YOLO_IOU = 0.45
BODY_CROP_PADDING = 0.12

MAX_IDS = 0
MAX_IMAGES_PER_ID = 0

# Registration
MIN_SINGLE_IMAGES_PER_ID = 2
REG_IMAGES_PER_ID = 3
MAX_REGISTERED_IDS = 300

# Leakage control
EXCLUDE_REG_FROM_GALLERY = True
EXCLUDE_REG_FROM_SYNTH_POOL = True

# Search defaults
DEFAULT_THRESHOLD = 0.50
DEFAULT_TOPK = 50

# Synthetic gallery
USE_SYNTHETIC_GALLERY = True
SYNTH_NUM_IMAGES = 4000
SYNTH_MAX_INSTANCES_PER_IMAGE = 4
UNKNOWN_ID_LABEL = '__UNKNOWN__'
UNKNOWN_POOL_RATIO = 0.25

# Cache
CACHE_ROOT = ROOT / 'notebooks' / 'embedding' / '.cache_multi_instance_demo_clean'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
USE_CACHE = True

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.jfif'}
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('DATASET_ROOT:', DATASET_ROOT, DATASET_ROOT.exists())
print('BODY_DET_WEIGHTS:', BODY_DET_WEIGHTS, BODY_DET_WEIGHTS.exists())
print('EMBED_LOCAL_MIEWID:', EMBED_LOCAL_MIEWID, EMBED_LOCAL_MIEWID.exists())


DATASET_ROOT: /data/dogfacenet/dataset/DogFaceNet_large/images_2 True
BODY_DET_WEIGHTS: /workspace/PoC/dogface_fastapi_poc_qdrant/weights/yolo/yolo26x.pt True
EMBED_LOCAL_MIEWID: /workspace/PoC/dogface_fastapi_poc_qdrant/weights/models--conservationxlabs--miewid-msv3 True


In [3]:
# Pretrained-only init
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

if PRETRAINED_ONLY:
    if MODEL_NAME.lower() != 'miewid':
        raise ValueError('This notebook is configured for local pretrained miewid only.')
    if not EMBED_LOCAL_MIEWID.exists():
        raise FileNotFoundError(f'Missing local miewid cache: {EMBED_LOCAL_MIEWID}')

settings = Settings()
settings.model_name = MODEL_NAME
settings.hf_cache_dir = EMBED_HF_ROOT
settings.device = DEVICE
embedder = Embedder(settings)
body_model = YOLO(str(BODY_DET_WEIGHTS), task='detect')

print('Embedder dim:', embedder.dim)
print('Device:', embedder.device)


Building Model Backbone for efficientnetv2_rw_m model
config.model_name efficientnetv2_rw_m
model_name efficientnetv2_rw_m
final_in_features 2152


/workspace/PoC/dogface_fastapi_poc_qdrant/.envqd/lib/python3.10/site-packages/torch/nn/modules/module.py:2586: UserWarning: for conv_stem.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/workspace/PoC/dogface_fastapi_poc_qdrant/.envqd/lib/python3.10/site-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/workspace/PoC/dogface_fastapi_poc_qdrant/.envqd/lib/python3.10/site-packages/torch/nn/modules/batchnorm.py:133: UserWa

Embedder dim: 2152
Device: cuda:0


In [4]:
# Utilities

def list_id_dirs(root: Path):
    return sorted([p for p in root.iterdir() if p.is_dir()], key=lambda x: x.name)


def list_images(d: Path):
    return sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS], key=lambda x: x.name)


def collect_dataset_samples(root: Path, max_ids=0, max_images_per_id=0):
    out = []
    id_dirs = list_id_dirs(root)
    if max_ids > 0:
        id_dirs = id_dirs[:max_ids]
    for d in id_dirs:
        imgs = list_images(d)
        if max_images_per_id > 0:
            imgs = imgs[:max_images_per_id]
        for p in imgs:
            out.append((d.name, p))
    return out


def image_uid(path: Path):
    st = path.stat()
    s = f"{path.resolve()}|{st.st_mtime_ns}|{st.st_size}"
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:16]


In [5]:
# Detection + embedding per image

def safe_open_rgb(path: Path):
    try:
        return Image.open(path).convert('RGB')
    except Exception:
        return None


def detect_and_embed_instances(image_path: Path, image_id: str):
    img = safe_open_rgb(image_path)
    if img is None:
        return {
            'image_uid': image_uid(image_path),
            'image_path': str(image_path),
            'image_id': image_id,
            'num_instances': 0,
            'instances': [],
        }

    arr = np.asarray(img)
    h, w = arr.shape[:2]

    res = body_model.predict(
        source=arr,
        imgsz=YOLO_IMG_SIZE,
        conf=YOLO_CONF,
        iou=YOLO_IOU,
        device=DEVICE,
        verbose=False,
    )

    if (not res) or (res[0].boxes is None) or (len(res[0].boxes) == 0):
        return {
            'image_uid': image_uid(image_path),
            'image_path': str(image_path),
            'image_id': image_id,
            'num_instances': 0,
            'instances': [],
        }

    r0 = res[0]
    boxes = r0.boxes.xyxy.cpu().numpy()
    confs = r0.boxes.conf.cpu().numpy().astype(float)

    crops = []
    metas = []
    for i, (b, c) in enumerate(zip(boxes, confs)):
        x1, y1, x2, y2 = b
        nb = NormalizedBBox(float(x1 / w), float(y1 / h), float(x2 / w), float(y2 / h))
        nb = pad_bbox(nb, BODY_CROP_PADDING)
        crop = crop_from_bbox(img, nb)
        crops.append(crop)
        metas.append({'inst_idx': int(i), 'bbox_xyxy': [float(x1), float(y1), float(x2), float(y2)], 'conf': float(c)})

    embs = embedder.embed_pil_images(crops)

    instances = []
    for i, m in enumerate(metas):
        instances.append({
            'inst_idx': m['inst_idx'],
            'bbox_xyxy': m['bbox_xyxy'],
            'conf': m['conf'],
            'embedding': embs[i].astype(np.float32),
        })

    return {
        'image_uid': image_uid(image_path),
        'image_path': str(image_path),
        'image_id': image_id,
        'num_instances': len(instances),
        'instances': instances,
    }


In [6]:
# Build gallery records

def build_gallery_records(samples):
    cfg = {
        'dataset_root': str(DATASET_ROOT),
        'max_ids': int(MAX_IDS),
        'max_images_per_id': int(MAX_IMAGES_PER_ID),
        'weights': str(BODY_DET_WEIGHTS),
        'model': MODEL_NAME,
        'imgsz': YOLO_IMG_SIZE,
        'conf': YOLO_CONF,
        'iou': YOLO_IOU,
        'pad': BODY_CROP_PADDING,
    }
    key = hashlib.sha1(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()[:12]
    cache_file = CACHE_ROOT / f'gallery_records_{key}.pkl'

    if USE_CACHE and cache_file.exists():
        with cache_file.open('rb') as f:
            recs = pickle.load(f)
        print('Loaded cache:', cache_file.name)
        return recs

    recs = []
    for image_id, p in tqdm(samples, desc='Build gallery records'):
        try:
            recs.append(detect_and_embed_instances(p, image_id))
        except Exception:
            continue

    if USE_CACHE:
        with cache_file.open('wb') as f:
            pickle.dump(recs, f)
        print('Saved cache:', cache_file.name)

    return recs


samples = collect_dataset_samples(DATASET_ROOT, MAX_IDS, MAX_IMAGES_PER_ID)
print('Samples:', len(samples))

gallery_records = build_gallery_records(samples)
print('Gallery images:', len(gallery_records))

num_instances = np.array([r['num_instances'] for r in gallery_records], dtype=np.int32)
print('Images >=1 instance:', int((num_instances >= 1).sum()))
print('Images >1 instances:', int((num_instances > 1).sum()))
print('Mean instances/image:', float(num_instances.mean()) if len(num_instances) else np.nan)


Samples: 11163


Build gallery records:   0%|          | 0/11163 [00:00<?, ?it/s]

Saved cache: gallery_records_0f8676d9f7c8.pkl
Gallery images: 11163
Images >=1 instance: 11113
Images >1 instances: 5734
Mean instances/image: 1.920809818149243


In [7]:
# Registration DB from single-object images

def build_registration_db(records, min_single_per_id=2, reg_per_id=3, max_registered_ids=300):
    single = [r for r in records if r['num_instances'] == 1]
    by_id = defaultdict(list)
    for r in single:
        by_id[r['image_id']].append(r)

    valid_ids = sorted([pid for pid, rows in by_id.items() if len(rows) >= min_single_per_id])
    if max_registered_ids > 0:
        valid_ids = valid_ids[:max_registered_ids]

    reg_db = {}
    reg_items = []
    for pid in valid_ids:
        rows = by_id[pid][:reg_per_id]
        reg_db[pid] = {
            'rows': rows,
            'embeddings': [rows[i]['instances'][0]['embedding'] for i in range(len(rows))],
        }
        reg_items.extend(rows)

    return reg_db, reg_items


reg_db, reg_items = build_registration_db(
    gallery_records,
    min_single_per_id=MIN_SINGLE_IMAGES_PER_ID,
    reg_per_id=REG_IMAGES_PER_ID,
    max_registered_ids=MAX_REGISTERED_IDS,
)
registered_ids = sorted(reg_db.keys())

REG_EXCLUDED_UIDS = set([r['image_uid'] for r in reg_items])
REG_EXCLUDED_PATHS = set([r['image_path'] for r in reg_items])

print('Registered IDs:', len(registered_ids))
print('Registration images:', len(reg_items))
print('Excluded UIDs:', len(REG_EXCLUDED_UIDS))


Registered IDs: 300
Registration images: 743
Excluded UIDs: 743


In [8]:
# Vector backends

def flatten_gallery_points(records, exclude_uids=None):
    exclude_uids = set(exclude_uids or [])
    pts = []
    for r in records:
        if r['image_uid'] in exclude_uids:
            continue
        for inst in r['instances']:
            pts.append({
                'vector': inst['embedding'].astype(np.float32),
                'payload': {
                    'image_uid': r['image_uid'],
                    'image_path': r['image_path'],
                    'image_id': r['image_id'],
                    'inst_idx': int(inst['inst_idx']),
                    'conf': float(inst['conf']),
                }
            })
    return pts


def build_matrix_backend(points):
    if not points:
        return {
            'E': np.zeros((0, 0), dtype=np.float32),
            'image_uid': np.array([], dtype=object),
            'image_id': np.array([], dtype=object),
            'image_path': np.array([], dtype=object),
        }
    E = np.stack([p['vector'] for p in points], axis=0).astype(np.float32)
    return {
        'E': E,
        'image_uid': np.array([p['payload']['image_uid'] for p in points], dtype=object),
        'image_id': np.array([p['payload']['image_id'] for p in points], dtype=object),
        'image_path': np.array([p['payload']['image_path'] for p in points], dtype=object),
    }


gallery_points_all = flatten_gallery_points(gallery_records)
matrix_backend_all = build_matrix_backend(gallery_points_all)

if EXCLUDE_REG_FROM_GALLERY:
    gallery_points_search = flatten_gallery_points(gallery_records, exclude_uids=REG_EXCLUDED_UIDS)
else:
    gallery_points_search = gallery_points_all

matrix_backend_search = build_matrix_backend(gallery_points_search)

print('Points all/search:', len(gallery_points_all), '/', len(gallery_points_search))


Points all/search: 21442 / 20699


In [9]:
# Retrieval core

def score_images_for_id(query_id, reg_db, backend, threshold=0.5, exclude_uids=None, exclude_paths=None):
    if query_id not in reg_db:
        return pd.DataFrame(columns=['image_uid', 'image_path', 'image_id', 'score'])

    exclude_uids = set(exclude_uids or [])
    exclude_paths = set(exclude_paths or [])

    q = np.stack(reg_db[query_id]['embeddings'], axis=0).astype(np.float32)
    E = backend['E']
    if E.size == 0:
        return pd.DataFrame(columns=['image_uid', 'image_path', 'image_id', 'score'])

    sims = q @ E.T
    max_sim_per_instance = sims.max(axis=0)

    out = {}
    for i, s in enumerate(max_sim_per_instance):
        uid = backend['image_uid'][i]
        ipath = backend['image_path'][i]
        if uid in exclude_uids or ipath in exclude_paths:
            continue
        prev = out.get(uid)
        if (prev is None) or (s > prev['score']):
            out[uid] = {
                'image_uid': uid,
                'image_path': ipath,
                'image_id': backend['image_id'][i],
                'score': float(s),
            }

    df = pd.DataFrame(list(out.values()))
    if len(df) == 0:
        return df
    return df[df['score'] >= float(threshold)].sort_values('score', ascending=False).reset_index(drop=True)


def search_by_id_list(selected_ids, reg_db, backend, threshold=0.5, mode='any', exclude_uids=None, exclude_paths=None):
    selected_ids = [sid for sid in selected_ids if sid in reg_db]
    if not selected_ids:
        return pd.DataFrame()

    base = None
    score_cols = []
    for sid in selected_ids:
        d = score_images_for_id(
            sid, reg_db, backend, threshold=-1.0,
            exclude_uids=exclude_uids,
            exclude_paths=exclude_paths,
        )
        if len(d) == 0:
            continue
        d = d.rename(columns={'score': f'score_{sid}'})
        score_cols.append(f'score_{sid}')
        keep = ['image_uid', 'image_path', 'image_id', f'score_{sid}']
        base = d[keep] if base is None else base.merge(d[keep], on=['image_uid', 'image_path', 'image_id'], how='outer')

    if base is None or len(base) == 0:
        return pd.DataFrame()

    base[score_cols] = base[score_cols].fillna(-1.0)

    if mode == 'all':
        mask = np.ones(len(base), dtype=bool)
        for c in score_cols:
            mask &= (base[c] >= threshold)
    else:
        mask = np.zeros(len(base), dtype=bool)
        for c in score_cols:
            mask |= (base[c] >= threshold)

    out = base[mask].copy()
    if len(out) == 0:
        return out
    out['score_max'] = out[score_cols].max(axis=1)
    return out.sort_values('score_max', ascending=False).reset_index(drop=True)


In [10]:
# Quick leakage sanity check
if registered_ids:
    qid = registered_ids[0]
    d_all = score_images_for_id(qid, reg_db, matrix_backend_all, threshold=DEFAULT_THRESHOLD)
    d_ex = score_images_for_id(
        qid, reg_db, matrix_backend_search, threshold=DEFAULT_THRESHOLD,
        exclude_uids=REG_EXCLUDED_UIDS if EXCLUDE_REG_FROM_GALLERY else None,
        exclude_paths=REG_EXCLUDED_PATHS if EXCLUDE_REG_FROM_GALLERY else None,
    )
    print('query_id:', qid)
    print('hits all/excluded:', len(d_all), '/', len(d_ex))
    display(d_all.head(5))
    display(d_ex.head(5))


query_id: 012_5
hits all/excluded: 6 / 3


,image_uid,image_path,image_id,score
0,ca14d25ad6edb7cf,/data/dogfacenet/dataset/DogFaceNet_large/imag...,012_5,1.000000
1,4ac16a1d549f53d6,/data/dogfacenet/dataset/DogFaceNet_large/imag...,012_5,1.000000
2,f9a5fa8e4114682d,/data/dogfacenet/dataset/DogFaceNet_large/imag...,012_5,1.000000
3,fe7491c9bf4cd323,/data/dogfacenet/dataset/DogFaceNet_large/imag...,70277-wunderschoene-tanja-w-1-j-such,0.527544
4,fdbb20a6666aede0,/data/dogfacenet/dataset/DogFaceNet_large/imag...,401241,0.501029


,image_uid,image_path,image_id,score
0,fe7491c9bf4cd323,/data/dogfacenet/dataset/DogFaceNet_large/imag...,70277-wunderschoene-tanja-w-1-j-such,0.527544
1,fdbb20a6666aede0,/data/dogfacenet/dataset/DogFaceNet_large/imag...,401241,0.501029
2,db0f909cb53ee9d6,/data/dogfacenet/dataset/DogFaceNet_large/imag...,Nozis,0.500894


In [11]:
# Synthetic multi-instance gallery

def single_instance_pool(records, exclude_uids=None):
    exclude_uids = set(exclude_uids or [])
    pool = []
    for r in records:
        if r['image_uid'] in exclude_uids:
            continue
        if r['num_instances'] == 1:
            inst = r['instances'][0]
            pool.append({
                'image_uid': r['image_uid'],
                'image_path': r['image_path'],
                'image_id': r['image_id'],
                'embedding': inst['embedding'],
            })
    return pool


def build_synthetic_gallery(pool, registered_ids, n_images=4000, max_instances=4, unknown_ratio=0.25, seed=42):
    rng = random.Random(seed)
    reg_set = set(registered_ids)

    reg_pool = [x for x in pool if x['image_id'] in reg_set]
    unknown_pool = [x for x in pool if x['image_id'] not in reg_set]
    if len(reg_pool) == 0:
        return []

    out = []
    for i in range(n_images):
        k = rng.randint(1, max_instances)
        picks = []
        use_unknown = (len(unknown_pool) > 0) and (rng.random() < unknown_ratio)

        if use_unknown:
            n_unk = rng.randint(1, max(1, k // 2))
            for _ in range(n_unk):
                picks.append(rng.choice(unknown_pool))
            for _ in range(k - n_unk):
                picks.append(rng.choice(reg_pool))
        else:
            for _ in range(k):
                picks.append(rng.choice(reg_pool))

        dedup = {}
        for p in picks:
            dedup[(p['image_uid'], p['image_id'])] = p
        picks = list(dedup.values())

        insts = []
        true_ids = set()
        for j, p in enumerate(picks):
            mapped = p['image_id'] if p['image_id'] in reg_set else UNKNOWN_ID_LABEL
            true_ids.add(mapped)
            insts.append({
                'inst_idx': j,
                'embedding': p['embedding'].astype(np.float32),
                'src_image_uid': p['image_uid'],
                'src_image_path': p['image_path'],
                'src_id': p['image_id'],
                'mapped_id': mapped,
            })

        out.append({
            'image_uid': f'synth_{i:06d}',
            'image_path': f'synth://{i:06d}',
            'image_id': UNKNOWN_ID_LABEL,
            'num_instances': len(insts),
            'instances': insts,
            'true_ids': sorted(true_ids),
        })

    return out


def flatten_synth_points(records, exclude_src_uids=None):
    exclude_src_uids = set(exclude_src_uids or [])
    pts = []
    for r in records:
        for inst in r['instances']:
            if inst.get('src_image_uid') in exclude_src_uids:
                continue
            pts.append({
                'vector': inst['embedding'].astype(np.float32),
                'payload': {
                    'image_uid': r['image_uid'],
                    'image_path': inst.get('src_image_path', r['image_path']),
                    'image_id': inst.get('mapped_id', UNKNOWN_ID_LABEL),
                    'inst_idx': int(inst['inst_idx'])
                }
            })
    return pts


if USE_SYNTHETIC_GALLERY:
    pool_ex = REG_EXCLUDED_UIDS if EXCLUDE_REG_FROM_SYNTH_POOL else set()
    pool = single_instance_pool(gallery_records, exclude_uids=pool_ex)
    synth_records = build_synthetic_gallery(
        pool,
        registered_ids,
        n_images=SYNTH_NUM_IMAGES,
        max_instances=SYNTH_MAX_INSTANCES_PER_IMAGE,
        unknown_ratio=UNKNOWN_POOL_RATIO,
        seed=SEED,
    )

    synth_points_all = flatten_synth_points(synth_records)
    synth_backend_all = build_matrix_backend(synth_points_all)

    if EXCLUDE_REG_FROM_GALLERY:
        synth_points_search = flatten_synth_points(synth_records, exclude_src_uids=REG_EXCLUDED_UIDS)
    else:
        synth_points_search = synth_points_all
    synth_backend_search = build_matrix_backend(synth_points_search)

    print('Synthetic images:', len(synth_records))
    print('Synth points all/search:', len(synth_points_all), '/', len(synth_points_search))
else:
    synth_records = []
    synth_backend_all = None
    synth_backend_search = None


Synthetic images: 4000
Synth points all/search: 9778 / 9778


In [12]:
# Synthetic metrics

def ap_at_ranking(y_true):
    y = np.asarray(y_true, dtype=np.int32)
    pos = int(y.sum())
    if pos == 0:
        return np.nan
    csum = np.cumsum(y)
    ranks = np.arange(1, len(y) + 1)
    prec = csum / ranks
    return float((prec * y).sum() / pos)


def evaluate_single_id_search_synth(reg_db, synth_records, synth_backend, threshold=0.5, ks=(1,5,10)):
    true_map = {r['image_uid']: set(r.get('true_ids', [])) for r in synth_records}
    rows = []

    for sid in tqdm(sorted(reg_db.keys()), desc='Eval single-ID synth'):
        d = score_images_for_id(
            sid, reg_db, synth_backend, threshold=-1.0,
            exclude_uids=REG_EXCLUDED_UIDS if EXCLUDE_REG_FROM_GALLERY else None,
            exclude_paths=REG_EXCLUDED_PATHS if EXCLUDE_REG_FROM_GALLERY else None,
        )
        if len(d) == 0:
            continue

        y = [1 if sid in true_map.get(uid, set()) else 0 for uid in d['image_uid'].tolist()]
        ap = ap_at_ranking(y)

        pred_pos = (d['score'].values >= threshold).astype(np.int32)
        y_true = np.asarray(y, dtype=np.int32)
        tp = int(((pred_pos == 1) & (y_true == 1)).sum())
        fp = int(((pred_pos == 1) & (y_true == 0)).sum())
        fn = int(((pred_pos == 0) & (y_true == 1)).sum())

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)

        row = {'id': sid, 'ap': ap, 'precision': precision, 'recall': recall, 'f1': f1}
        for k in ks:
            row[f'top{k}'] = float(np.mean(y[:k])) if len(y) >= k else float(np.mean(y))
        rows.append(row)

    return pd.DataFrame(rows)


def summarize_macro(df):
    if df is None or len(df) == 0:
        return {'macro_ap': np.nan, 'macro_precision': np.nan, 'macro_recall': np.nan, 'macro_f1': np.nan}
    return {
        'macro_ap': float(df['ap'].mean()),
        'macro_precision': float(df['precision'].mean()),
        'macro_recall': float(df['recall'].mean()),
        'macro_f1': float(df['f1'].mean()),
    }


if USE_SYNTHETIC_GALLERY and synth_backend_all is not None and synth_backend_search is not None:
    eval_single_all = evaluate_single_id_search_synth(reg_db, synth_records, synth_backend_all, threshold=DEFAULT_THRESHOLD)
    eval_single_ex = evaluate_single_id_search_synth(reg_db, synth_records, synth_backend_search, threshold=DEFAULT_THRESHOLD)

    cmp = pd.DataFrame([
        {'setting':'with_leakage(all)', **summarize_macro(eval_single_all)},
        {'setting':'without_leakage(excluded)', **summarize_macro(eval_single_ex)},
    ])
    display(cmp)
else:
    eval_single_all = pd.DataFrame()
    eval_single_ex = pd.DataFrame()


Eval single-ID synth:   0%|          | 0/300 [00:00<?, ?it/s]

Eval single-ID synth:   0%|          | 0/300 [00:00<?, ?it/s]

,setting,macro_ap,macro_precision,macro_recall,macro_f1
0,with_leakage(all),0.851298,0.085074,0.09009,0.08677
1,without_leakage(excluded),0.851298,0.085074,0.09009,0.08677


In [13]:
# Interactive demo (clean)
DET_VIS_CACHE = {}


def _safe_open_rgb(path_str):
    try:
        return Image.open(path_str).convert('RGB')
    except Exception:
        return None


def _detect_boxes_cached(path_str):
    key = f"{path_str}|{YOLO_IMG_SIZE}|{YOLO_CONF}|{YOLO_IOU}|{DEVICE}"
    if key in DET_VIS_CACHE:
        return DET_VIS_CACHE[key]

    img = _safe_open_rgb(path_str)
    if img is None:
        DET_VIS_CACHE[key] = []
        return []

    arr = np.asarray(img)
    res = body_model.predict(source=arr, imgsz=YOLO_IMG_SIZE, conf=YOLO_CONF, iou=YOLO_IOU, device=DEVICE, verbose=False)
    if (not res) or (res[0].boxes is None) or (len(res[0].boxes) == 0):
        DET_VIS_CACHE[key] = []
        return []

    boxes = res[0].boxes.xyxy.cpu().numpy().tolist()
    DET_VIS_CACHE[key] = boxes
    return boxes


def _draw_image(ax, path_str, title='', det_mode='stored_only', bboxes=None, max_det_boxes=10):
    img = _safe_open_rgb(path_str)
    if img is None:
        ax.text(0.5, 0.5, 'read_fail', ha='center', va='center')
        ax.set_title(title, fontsize=9)
        ax.axis('off')
        return

    arr = np.asarray(img)
    ax.imshow(arr)

    draw_boxes = []
    if det_mode == 'off':
        draw_boxes = []
    elif det_mode == 'stored_only':
        draw_boxes = bboxes if bboxes is not None else []
    elif det_mode == 'run_model':
        draw_boxes = bboxes if bboxes is not None else _detect_boxes_cached(path_str)

    for i, b in enumerate(draw_boxes[:max_det_boxes]):
        x1, y1, x2, y2 = b
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='lime', linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), f'#{i}', color='lime', fontsize=8, backgroundcolor='black')

    ax.set_title(title, fontsize=9)
    ax.axis('off')


def _draw_composite(ax, comp_img, boxes, labels, title=''):
    ax.imshow(np.asarray(comp_img))
    for i, (b, lab) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = b
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='cyan', linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), f'{i}:{lab}', color='cyan', fontsize=8, backgroundcolor='black')
    ax.set_title(title, fontsize=10)
    ax.axis('off')


def _rows_to_axes(n_items, cols, fig_scale=4.5):
    cols = max(1, int(cols))
    rows = max(1, (n_items + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(fig_scale * cols, fig_scale * rows * 0.85))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    return fig, axes


def _pick_backend_lookup():
    if USE_SYNTHETIC_GALLERY and synth_backend_search is not None and len(synth_records) > 0:
        return synth_backend_search, {r['image_uid']: r for r in synth_records}, 'synthetic'
    return matrix_backend_search, {r['image_uid']: r for r in gallery_records}, 'real'


def _find_multi_single(df, lookup):
    m = None
    s = None
    for _, row in df.iterrows():
        rec = lookup.get(row['image_uid'])
        if rec is None:
            continue
        ni = int(rec.get('num_instances', 0))
        if ni > 1 and m is None:
            m = row
        if ni == 1 and s is None:
            s = row
        if m is not None and s is not None:
            break
    return m, s


def _build_synth_composite(rec, cols=4, tile_w=320, tile_h=220, pad=8):
    insts = rec.get('instances', [])
    n = len(insts)
    cols = max(1, int(cols))
    rows = max(1, (n + cols - 1) // cols)

    W = cols * tile_w + (cols + 1) * pad
    H = rows * tile_h + (rows + 1) * pad
    canvas = Image.new('RGB', (W, H), (24, 24, 24))

    boxes, labels = [], []
    for i, inst in enumerate(insts):
        r = i // cols
        c = i % cols
        x0 = pad + c * (tile_w + pad)
        y0 = pad + r * (tile_h + pad)

        p = inst.get('src_image_path', rec.get('image_path'))
        im = _safe_open_rgb(p)
        patch = Image.new('RGB', (tile_w, tile_h), (50, 50, 50)) if im is None else im.resize((tile_w, tile_h), Image.BILINEAR)
        canvas.paste(patch, (x0, y0))

        boxes.append((x0, y0, x0 + tile_w, y0 + tile_h))
        labels.append(f"{inst.get('src_id','NA')}->{inst.get('mapped_id','NA')}")

    return canvas, boxes, labels


def render_demo(query_id, threshold, topk, reg_cols, multi_cols, fig_scale, det_mode, max_det_boxes):
    if query_id is None or query_id not in reg_db:
        print('Invalid query_id')
        return

    backend, lookup, backend_name = _pick_backend_lookup()
    df = score_images_for_id(
        query_id,
        reg_db,
        backend,
        threshold=float(threshold),
        exclude_uids=REG_EXCLUDED_UIDS if EXCLUDE_REG_FROM_GALLERY else None,
        exclude_paths=REG_EXCLUDED_PATHS if EXCLUDE_REG_FROM_GALLERY else None,
    ).head(int(topk))

    if len(df) == 0:
        print(f'No hits: query={query_id}, threshold={threshold:.2f}')
        return

    m_row, s_row = _find_multi_single(df, lookup)
    m_rec = lookup.get(m_row['image_uid']) if m_row is not None else None
    s_rec = lookup.get(s_row['image_uid']) if s_row is not None else None

    # registration panel
    reg_rows = reg_db[query_id]['rows']
    fig_r, ax_r = _rows_to_axes(len(reg_rows), reg_cols, fig_scale)
    flat = ax_r.flatten()
    for i, rr in enumerate(reg_rows):
        p = rr['image_path']
        bbs = [inst.get('bbox_xyxy') for inst in rr.get('instances', []) if 'bbox_xyxy' in inst]
        _draw_image(flat[i], p, title=f'reg[{i}]\n{Path(p).name}', det_mode=det_mode, bboxes=bbs if bbs else None, max_det_boxes=max_det_boxes)
    for i in range(len(reg_rows), len(flat)):
        flat[i].axis('off')
    fig_r.suptitle(f'Registration (single-object) | query={query_id}', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

    # multi panel
    if m_rec is None:
        print('No multi-instance hit in current topK.')
    else:
        if backend_name == 'synthetic':
            comp, boxes, labels = _build_synth_composite(m_rec, cols=multi_cols)
            fig_m, ax_m = plt.subplots(1, 1, figsize=(fig_scale * max(2, multi_cols), fig_scale * 1.3))
            _draw_composite(ax_m, comp, boxes, labels, title='synthetic composed image')
        else:
            fig_m, ax_m = _rows_to_axes(1, 1, fig_scale)
            bbs = [inst.get('bbox_xyxy') for inst in m_rec.get('instances', []) if 'bbox_xyxy' in inst]
            _draw_image(ax_m.flatten()[0], m_rec['image_path'], title='retrieved multi image', det_mode=det_mode, bboxes=bbs if bbs else None, max_det_boxes=max_det_boxes)
        fig_m.suptitle(f"Retrieved Multi | uid={m_rec['image_uid']} | score={float(m_row['score']):.3f} | backend={backend_name}", fontsize=12, fontweight='bold')
        plt.tight_layout(); plt.show()

    # single panel
    if s_rec is None:
        print('No single-instance hit in current topK.')
    else:
        if backend_name == 'synthetic':
            comp, boxes, labels = _build_synth_composite(s_rec, cols=1)
            fig_s, ax_s = plt.subplots(1, 1, figsize=(fig_scale * 1.8, fig_scale * 1.1))
            _draw_composite(ax_s, comp, boxes, labels, title='synthetic single composed image')
        else:
            fig_s, ax_s = _rows_to_axes(1, 1, fig_scale)
            bbs = [inst.get('bbox_xyxy') for inst in s_rec.get('instances', []) if 'bbox_xyxy' in inst]
            _draw_image(ax_s.flatten()[0], s_rec['image_path'], title='retrieved single image', det_mode=det_mode, bboxes=bbs if bbs else None, max_det_boxes=max_det_boxes)
        fig_s.suptitle(f"Retrieved Single | uid={s_rec['image_uid']} | score={float(s_row['score']):.3f}", fontsize=12, fontweight='bold')
        plt.tight_layout(); plt.show()

    display(df.head(10))


if len(registered_ids) == 0:
    print('No registered_ids. Run previous cells first.')
else:
    w_query = widgets.Dropdown(options=registered_ids, value=registered_ids[0], description='Query ID:', layout=widgets.Layout(width='420px'))
    w_th = widgets.FloatSlider(value=float(DEFAULT_THRESHOLD), min=0.0, max=1.0, step=0.01, description='Threshold:', readout_format='.2f', continuous_update=False)
    w_topk = widgets.IntSlider(value=DEFAULT_TOPK, min=1, max=200, step=1, description='TopK:', continuous_update=False)
    w_reg_cols = widgets.IntSlider(value=3, min=1, max=8, step=1, description='Reg Cols:', continuous_update=False)
    w_mul_cols = widgets.IntSlider(value=4, min=1, max=8, step=1, description='Multi Cols:', continuous_update=False)
    w_fig = widgets.FloatSlider(value=4.8, min=2.5, max=9.0, step=0.1, description='Fig Scale:', readout_format='.1f', continuous_update=False)
    w_det_mode = widgets.Dropdown(
        options=[('Off','off'), ('Stored bbox only (fast)','stored_only'), ('Run detector overlay (slow)','run_model')],
        value='stored_only',
        description='Det Overlay:',
        layout=widgets.Layout(width='360px')
    )
    w_max_det = widgets.IntSlider(value=8, min=1, max=30, step=1, description='Max bbox:', continuous_update=False)

    btn = widgets.Button(description='Render Demo', button_style='info', icon='image')
    out = widgets.Output()

    ui = widgets.VBox([
        widgets.HTML('<b>Interactive Demo Controls</b>'),
        widgets.HBox([w_query, w_det_mode]),
        widgets.HBox([w_th, w_topk]),
        widgets.HBox([w_reg_cols, w_mul_cols, w_fig, w_max_det]),
        btn,
    ])

    def _run():
        with out:
            out.clear_output(wait=True)
            render_demo(
                query_id=w_query.value,
                threshold=w_th.value,
                topk=w_topk.value,
                reg_cols=w_reg_cols.value,
                multi_cols=w_mul_cols.value,
                fig_scale=w_fig.value,
                det_mode=w_det_mode.value,
                max_det_boxes=w_max_det.value,
            )

    btn.on_click(lambda _: _run())
    display(ui, out)
    _run()


Output()

In [14]:
# Save key outputs
RUN_TAG = time.strftime('%Y%m%d_%H%M%S')
OUT_DIR = ROOT / 'notebooks' / 'embedding' / 'results' / f'multi_instance_demo_clean_{RUN_TAG}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

meta = {
    'n_samples': len(samples),
    'n_gallery_images': len(gallery_records),
    'n_gallery_instances': int(sum(r['num_instances'] for r in gallery_records)),
    'n_registered_ids': len(registered_ids),
    'n_registration_images': len(reg_items),
    'exclude_reg_from_gallery': EXCLUDE_REG_FROM_GALLERY,
    'exclude_reg_from_synth_pool': EXCLUDE_REG_FROM_SYNTH_POOL,
}
pd.DataFrame([meta]).to_csv(OUT_DIR / 'meta_summary.csv', index=False)

if isinstance(eval_single_all, pd.DataFrame) and len(eval_single_all):
    eval_single_all.to_csv(OUT_DIR / 'eval_single_synth_with_leakage.csv', index=False)
if isinstance(eval_single_ex, pd.DataFrame) and len(eval_single_ex):
    eval_single_ex.to_csv(OUT_DIR / 'eval_single_synth_without_leakage.csv', index=False)

print('Saved:', OUT_DIR)


Saved: /workspace/PoC/dogface_fastapi_poc_qdrant/notebooks/embedding/results/multi_instance_demo_clean_20260210_085556


## Execution Order

Run top-to-bottom once. If you change configuration, rerun from the configuration cell.
